# 图像卷积

## 一些基本概念
- 卷积时，第一层输入的通道数一般是RGB=3（而中间层则没有这个限制）,输出的通道数与卷积核的个数相同。-
- 每个卷积核作用的结果是将输入的3个通道信息“加权融合”到1个输出通道，
- N个卷积核将得到N个输出通道。
- 不同卷积核的作用是“提取不同的特征”--因此卷积核也是“特征提取器”。
- 不同卷积核大小不一定完全相同。
- 卷积核是通过训练得到的

## 常见参数：
- 卷积核大小、维度、Batch
- Padding(填充)
- Stride（计算步长）
- Dilation(膨胀率)

### 卷积核大小选择策略

| 大小 | 名称 | 感受野 | 特点 | 适用场景 |
|------|------|--------|------|----------|
| **1×1** | 点卷积 | 1像素 | 通道融合，降维/升维 | 网络inception结构，通道调整 |
| **3×3** | 标准卷积 | 3×3区域 | 平衡计算量和感受野 | 最常用，VGG/ResNet等主流网络 |
| **5×5** | 较大卷积 | 5×5区域 | 较大感受野，计算量较大 | 需要捕捉更大范围特征 |
| **7×7** | 大卷积 | 7×7区域 | 感受野大，计算量大 | 早期网络，输入层 |
| **非对称** | 如1×3, 3×1 | 矩形区域 | 减少参数，特定方向特征 | 文本识别，特定方向特征提取 |

### 关键原则：
1. **同一卷积层**：卷积核大小**通常相同**
2. **不同卷积层**：卷积核大小**可以不同**
3. **多尺度特征提取**：使用不同大小的卷积核并行处理

```
# 中间层的输入通道数不一定是3(特例：如果是RGB的输入，则是3)
# 输入通道=上一层的输出通道数=C_i
# 每个卷积核：形状为 (k, k, C_i)
# N个卷积核：产生N个输出通道
# 考虑并行计算：产生batch的概念:B
```
因此，卷积计算公式 $输入*卷积核=输出$：
$$[B, H_i, W_i, C_i]*[C_o, K_h, K_w, C_i]=[B,H_o, W_o, C_o]$$

### 卷积核的"特征提取器"角色

#### 不同类型卷积核提取的特征：

| 卷积核类型 | 提取的特征 | 可视化表现 | 在神经网络中的作用 |
|------------|------------|------------|-------------------|
| **边缘检测核** | 边缘、轮廓 | 线条、边界 | 初级特征，物体形状 |
| **纹理检测核** | 纹理、图案 | 重复模式、纹理 | 中级特征，表面特性 |
| **颜色检测核** | 颜色分布 | 色彩区域 | 颜色信息提取 |
| **组合特征核** | 复杂模式 | 特定形状、部件 | 高级特征，物体部件 |
| **专用特征核** | 特定特征 | 眼睛、车轮等 | 最终分类/识别 |

### 从浅层到深层的特征提取过程：

"""
Layer 1 (浅层) 卷积核：
├── 水平边缘检测器
├── 垂直边缘检测器
├── 45°边缘检测器
├── 颜色边缘检测器
└── ... (其他基本特征)

Layer 2 (中层) 卷积核：
├── 纹理检测器（条纹、斑点）
├── 角点检测器
├── 简单形状检测器（圆形、方形）
└── 组合边缘检测器

Layer 3+ (深层) 卷积核：
├── 物体部件检测器（眼睛、轮子）
├── 完整物体检测器（人脸、汽车）
└── 复杂模式识别器
"""

## EXAMPLE

In [ ]:
import torch
import torch.nn as nn

# 创建一个卷积层
conv_layer = nn.Conv2d(
    in_channels=3,      # 输入通道数：3 (RGB)
    out_channels=64,    # 输出通道数：64 = 64个卷积核
    kernel_size=3,      # 卷积核大小：3×3
    padding=1           # 保持空间尺寸
)

"""
工作原理：
1. 输入：一个3通道的RGB图像 (H, W, 3)
2. 卷积核：有64个不同的3×3×3卷积核
3. 每个卷积核：
   - 与输入的3个通道分别卷积
   - 将3个通道的结果相加
   - 得到一个单通道的输出特征图
4. 总输出：64个不同的特征图 (H, W, 64)

每个卷积核学习提取一种特定的特征：
- 卷积核1：可能学习检测水平边缘
- 卷积核2：可能学习检测垂直边缘  
- 卷积核3：可能学习检测红色区域
- ...
- 卷积核64：可能学习检测特定纹理

通过组合这些特征，网络能识别复杂的模式。
"""

## Padding

### 一、什么是Padding？
在卷积神经网络中，**Padding**是在输入特征图的**边界周围添加额外的像素（通常是0）**，以控制输出特征图的尺寸和保留边界信息。
- 例如，对于3x3卷积，会使得输出的维度长宽各自减少1. 对于5x5卷积，会使得输出的维度长宽各自减少2.

### 二、主要作用与目的

| 作用 | 说明 | 图示示例 |
|------|------|----------|
| **尺寸保持** | 使输出特征图与输入尺寸相同 | 输入5×5 → 卷积 → 输出5×5 |
| **边界信息保留** | 防止边界特征在卷积中被忽略 | 边界像素也能被充分卷积 |
| **感受野控制** | 更有效地利用大感受野 | 深层网络能获取更全局信息 |

### 三、Padding的类型对比

| 类型 | 公式 | 特点 | 适用场景 |
|------|------|------|----------|
| **Valid Padding**<br>(无填充) | `output_size = (n - f + 1)` | • 不添加填充<br>• 输出尺寸减小<br>• TensorFlow默认 | 需要减少特征图尺寸时 |
| **Same Padding**<br>(相同填充) | `output_size = n` | • 添加填充使输出尺寸不变<br>• Keras/PyTorch常用<br>• 计算padding值 | 需要保持特征图尺寸时 |
| **Full Padding** | `output_size = (n + f - 1)` | • 最大填充<br>• 输出尺寸增大<br>• 不常用 | 特定信号处理场景 |

### 四、Padding尺寸计算公式

#### 1. 输出尺寸通用公式：

输出高度 = (输入高度 + 2×padding高 - 卷积核高) / 步长 + 1
输出宽度 = (输入宽度 + 2×padding宽 - 卷积核宽) / 步长 + 1

#### 2. 计算所需Padding值（使输出尺寸不变）：

padding = floor((卷积核尺寸 - 1) / 2)

- 对于3×3卷积核：`padding = floor((3-1)/2) = 1`
- 对于5×5卷积核：`padding = floor((5-1)/2) = 2`

### 五、常见卷积核的Padding值

| 卷积核大小 | 保持尺寸的Padding | 说明 |
|------------|-------------------|------|
| 1×1 | 0 | 点卷积，无需填充 |
| 3×3 | 1 | 最常用，每边加1像素 |
| 5×5 | 2 | 每边加2像素 |
| 7×7 | 3 | 每边加3像素 |

### 六、代码示例



In [ ]:
import torch
import torch.nn as nn

import numpy as np

# PyTorch示例
conv_torch = nn.Conv2d(
    in_channels=3,
    out_channels=64,
    kernel_size=3,
    padding=1,          # Same padding for 3x3 kernel
    stride=1
)
# 输入尺寸不变：H × W → H × W

# 不同padding设置的效果
input_size = 5
kernel_size = 3

# 1. 无padding (Valid)
output_valid = input_size - kernel_size + 1  # 5-3+1=3

# 2. Same padding (保持尺寸)
padding = (kernel_size - 1) // 2  # (3-1)//2=1
output_same = (input_size + 2*padding - kernel_size) + 1  # (5+2-3)+1=5

print(f"输入尺寸: {input_size}×{input_size}")
print(f"Valid Padding输出: {output_valid}×{output_valid}")
print(f"Same Padding输出: {output_same}×{output_same}")

七、Padding对网络的影响

```markdown
| 影响维度 | 有Padding（如Same Padding） | 无Padding（Valid Padding） |
|---------|--------------------------|-------------------------|
| **特征图尺寸** | 保持不变或可控减小 | 每层快速减小 |
| **边界信息利用** | 充分保留，边界像素也能多次参与卷积 | 逐渐丢失，边界像素参与次数少 |
| **网络深度限制** | 可设计更深网络（几十到上百层） | 深度受限（几层后尺寸就太小） |
| **参数效率** | 相对较低（相同尺寸需要更多层） | 相对较高（快速减少计算量） |
| **计算量** | 较大（保持大尺寸特征图） | 较小（特征图快速缩小） |
| **内存占用** | 较高（保持大特征图） | 较低（特征图快速缩小） |
| **感受野增长** | 较慢（需要更多层达到大感受野） | 较快（尺寸缩小快，相对感受野增长快） |
| **信息保留** | 更好，适合密集预测任务 | 较差，可能丢失细节 |
| **常见应用** | 图像分割、目标检测、语义分割 | 图像分类（后期可配合池化） |
| **特征提取质量** | 更全面的空间信息 | 可能丢失边缘特征 |
| **收敛速度** | 相对较慢（参数多） | 相对较快（参数少） |
| **过拟合风险** | 较高（参数多） | 较低（参数少） |
| **示例配置** | `kernel=3, padding=1, stride=1` | `kernel=3, padding=0, stride=1` |
| **输出尺寸公式** | `输出 = (输入 + 2×pad - 核)/步长 + 1` | `输出 = (输入 - 核)/步长 + 1` |
| **实际网络案例** | U-Net（分割）、YOLO（检测） | 早期LeNet、简单分类网络 |

```

八、实际应用建议
常用配置：

text
kernel_size=3, padding=1, stride=1  # 保持尺寸
kernel_size=3, padding=0, stride=2  # 下采样
选择策略：

分类任务：早期层可用stride=2下采样，后期用padding保持尺寸

分割/检测任务：多用padding保持高分辨率特征图

轻量化网络：可减少padding以降低计算量

注意问题：

信息泄露：零填充可能引入边界伪影

计算开销：padding增加计算量

对称性：确保padding对称，避免特征偏移

九、可视化理解
```text
原始输入 (5×5):
┌───┬───┬───┬───┬───┐
│ 1 │ 2 │ 3 │ 4 │ 5 │
├───┼───┼───┼───┼───┤
│ 6 │ 7 │ 8 │ 9 │10 │
├───┼───┼───┼───┼───┤
│11 │12 │13 │14 │15 │
├───┼───┼───┼───┼───┤
│16 │17 │18 │19 │20 │
├───┼───┼───┼───┼───┤
│21 │22 │23 │24 │25 │
└───┴───┴───┴───┴───┘
```
添加Padding=1后 (7×7):
```text
┌───┬───┬───┬───┬───┬───┬───┐
│ 0 │ 0 │ 0 │ 0 │ 0 │ 0 │ 0 │
├───┼───┼───┼───┼───┼───┼───┤
│ 0 │ 1 │ 2 │ 3 │ 4 │ 5 │ 0 │
├───┼───┼───┼───┼───┼───┼───┤
│ 0 │ 6 │ 7 │ 8 │ 9 │10 │ 0 │
├───┼───┼───┼───┼───┼───┼───┤
│ 0 │11 │12 │13 │14 │15 │ 0 │
├───┼───┼───┼───┼───┼───┼───┤
│ 0 │16 │17 │18 │19 │20 │ 0 │
├───┼───┼───┼───┼───┼───┼───┤
│ 0 │21 │22 │23 │24 │25 │ 0 │
├───┼───┼───┼───┼───┼───┼───┤
│ 0 │ 0 │ 0 │ 0 │ 0 │ 0 │ 0 │
└───┴───┴───┴───┴───┴───┴───┘
```
十、一句话总结
Padding是通过在输入边界添加像素（通常为0）来控制卷积输出尺寸、保留边界信息的关键技术，常用的Same Padding能保持特征图尺寸不变，是构建深层CNN的基础。

## 计算步长
### 一、核心定义

**步长（Stride）** 是卷积核在输入特征图上**每次滑动的像素距离**。

- **默认值**：stride=1（每次移动1像素）
- **作用**：控制输出特征图的**下采样率**和**感受野**

### 二、基本工作原理
- Stride = 1， 就是一个像素一个像素的计算
- Stride = 2， 就是计算一个像素，然后跳过一个像素，下一次针对第三个像素做计算
- 以此类推

### 计算输出像素尺寸
输出尺寸 = ⌊(输入尺寸 + 2×padding - 卷积核尺寸) / stride⌋ + 1

## 四、不同步长效果对比

| 步长值 | 输出尺寸 | 计算量 | 感受野 | 信息保留 | 主要用途 |
|--------|----------|--------|--------|----------|----------|
| **stride=1** | 大（接近输入） | 大 | 小 | 完整 | 特征提取，保持分辨率 |
| **stride=2** | 减半 | 减少75% | 增大 | 有损 | 下采样，常用配置 |
| **stride>2** | 显著减小 | 大幅减少 | 大 | 损失多 | 快速下采样，轻量化 |

## 五、实际应用示例




In [ ]:
import torch.nn as nn

# 示例1：保持分辨率的特征提取
conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1)
# 输入224×224 → 输出224×224

# 示例2：下采样层
conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
# 输入224×224 → 输出112×112

# 示例3：替代池化的下采样
conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1)
# 输入112×112 → 输出56×56

### 六、步长与感受野关系

感受野 = 1 + Σ[(kernel_size_i - 1) × Π(stride_j)] (j=1 to i-1)

- stride=1：感受野线性增长
- stride=2：感受野指数增长
- 大stride：快速获得全局信息

### 七、设计选择建议
- 分类网络：
  早期：stride=2下采样
  后期：stride=1保持特征图尺寸

- 密集预测任务（分割/检测）：
  编码器：stride=2下采样
  解码器：stride=1或转置卷积上采样

- 轻量化网络：
  使用更大stride减少计算量
  结合深度可分离卷积

### 八、注意事项
- 信息丢失：大stride会跳过细节特征

- 网格效应：stride>1可能产生棋盘效应

- 尺寸匹配：确保输出尺寸为整数

- 替代方案：可用空洞卷积增大感受野而不增加stride

### 九、一句话总结
-步长控制卷积核滑动距离：stride=1保持分辨率用于特征提取，stride>1实现下采样减少计算量，是平衡计算效率与特征保持的关键超参数。

## 膨胀率
### 一、核心定义

**膨胀率（Dilation Rate）** 是卷积核元素之间的**间隔距离**，也称为"空洞率"或"扩张率"。

- **标准卷积**：dilation=1（元素紧密相邻）
- **空洞卷积**：dilation>1（元素间有间隔）
- **目的**：**增大感受野而不增加参数数量或降低分辨率**

### 二、工作原理对比
- **标准卷积**：dilation=1（元素紧密相邻）
- *空洞卷积**：dilation=2, 卷积时使用的输入像素点是每隔一个像素取用--相当于将输入的图像“放大”了2倍（**等效于增大感受野**）。这次被忽略的像素点在下一次卷积运算会被用到。

有效感受野尺寸 = (kernel_size - 1) × dilation + 1


### 示例计算：
| 核大小 | 膨胀率 | 有效感受野 | 参数数量 |
|--------|---------|------------|----------|
| 3×3 | 1 | 3×3 | 9 |
| 3×3 | 2 | 5×5 | 9 |
| 3×3 | 3 | 7×7 | 9 |
| 3×3 | 4 | 9×9 | 9 |

**关键优势**：感受野指数增长，但**参数数量不变**！

## 四、与步长（Stride）的对比

| 特性 | 膨胀卷积（Dilation） | 大stride卷积 |
|------|-------------------|--------------|
| **主要目的** | 增大感受野 | 下采样 |
| **输出分辨率** | **保持不变** | 降低 |
| **参数数量** | 不变 | 不变 |
| **信息完整性** | 保持密集采样 | 跳跃采样，信息丢失 |
| **计算量** | 不变（但稀疏计算） | 减少 |
| **感受野增长** | 指数增长 | 线性增长 |

## 五、膨胀卷积的数学表达

对于3×3卷积核，膨胀率=r：

实际覆盖位置 = 标准位置 × r
示例（r=2）：
标准位置：(0,0), (0,1), (0,2)...
膨胀位置：(0,0), (0,2), (0,4)...

In [ ]:
## 六、代码示例

import torch
import torch.nn as nn
import torch.nn.functional as F

# PyTorch实现
class DilatedConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        # 不同膨胀率的卷积层
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, dilation=1, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, dilation=2, padding=2)  # padding = dilation
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, dilation=4, padding=4)
        self.conv4 = nn.Conv2d(256, 512, kernel_size=3, dilation=8, padding=8)
        
        # padding计算：padding = dilation × (kernel_size - 1) // 2
        
    def forward(self, x):
        # 所有层保持相同分辨率
        x1 = F.relu(self.conv1(x))  # 感受野: 3×3
        x2 = F.relu(self.conv2(x1)) # 感受野: 7×7
        x3 = F.relu(self.conv3(x2)) # 感受野: 15×15
        x4 = F.relu(self.conv4(x3)) # 感受野: 31×31
        
        return x4

# # TensorFlow/Keras示例
# """
# from tensorflow.keras.layers import Conv2D

# # 空洞卷积在Keras中
# conv = Conv2D(filters=64, kernel_size=3, dilation_rate=2, padding='same')
# # dilation_rate可以是整数或元组 (row_dilation, col_dilation)
# """

### 十、优点与缺点
✅ 优点：
- 大感受野，少参数：指数级感受野增长，参数恒定
- 保持分辨率：不降低特征图尺寸
- 密集预测友好：适合分割、检测等任务
- 多尺度特征：容易实现空间金字塔

❌ 缺点：
- 网格效应：dilation过大时，卷积核不连续
- 局部信息丢失：跳过中间像素，可能漏掉细节
- 计算优化差：稀疏计算不利于硬件加速
- 训练不稳定：过大dilation可能导致梯度问题

# 手搓卷积

In [ ]:
import numpy as np


def Visualization(img):
    """
    可视化图像的不同通道。

    参数:
    img (numpy.ndarray): 输入的多通道图像。
    """
    import matplotlib.pyplot as plt  # 导入 Matplotlib 库

    # 创建子图布局
    fig, ax = plt.subplots(1, 4, figsize=(12, 16))
    # 可视化图像的特定通道
    ax[0].matshow(img[:, :, 0], cmap="viridis")  # 显示第一个通道
    ax[1].matshow(img[:, :, 5], cmap="viridis")  # 显示第六个通道
    ax[2].matshow(img[:, :, 11], cmap="viridis")  # 显示第十二个通道
    ax[3].matshow(img[:, :, 24], cmap="viridis")  # 显示第二十五个通道

    plt.savefig("a.png")  # 将可视化结果保存为图片


def Conv2d(img, weight, hi, wi, ci, co, kernel, stride, pad):
    """
    执行2D卷积操作。

    参数:
    img (numpy.ndarray): 输入图像。
    weight (list): 卷积核的权重。
    hi, wi (int): 输入图像的高度和宽度。
    ci, co (int): 输入和输出通道数。
    kernel (int): 卷积核大小。
    stride (int): 卷积步长。
    pad (int): 边缘填充大小。

    返回:
    numpy.ndarray: 卷积后的输出图像。
    """
    # 计算输出图像的尺寸
    ho = (hi + 2 * pad - kernel) // stride + 1
    wo = (wi + 2 * pad - kernel) // stride + 1

    # 转换权重格式并对图像进行填充
    weight = np.array(weight).reshape(co, kernel, kernel, ci)
    img_pad = np.pad(img, ((pad, pad), (pad, pad), (0, 0)), "constant")
    img_out = np.zeros((ho, wo, co))

    # 执行卷积操作
    for co_ in range(co):
        for ho_ in range(ho):
            in_h_origin = ho_ * stride - pad
            for wo_ in range(wo):
                in_w_origin = wo_ * stride - pad
                # 对每个卷积窗口进行计算
                filter_h_start = max(0, -in_h_origin)
                filter_w_start = max(0, -in_w_origin)
                filter_h_end = min(kernel, hi - in_h_origin)
                filter_w_end = min(kernel, wi - in_w_origin)
                acc = float(0)
                for kh_ in range(filter_h_start, filter_h_end):
                    hi_index = in_h_origin + kh_
                    for kw_ in range(filter_w_start, filter_w_end):
                        wi_index = in_w_origin + kw_
                        for ci_ in range(ci):
                            in_data = img[hi_index][wi_index][ci_]
                            weight_data = weight[co_][kh_][kw_][ci_]
                            acc = acc + in_data * weight_data
                img_out[ho_][wo_][co_] = acc
    return img_out  # 返回卷积后的图像


def Conv2dOpt(img, weight, hi, wi, ci, co, kernel, stride, pad):
    """
    执行2D卷积操作，使用优化技巧提高性能。

    参数:
    同 Conv2d 函数。

    返回:
    numpy.ndarray: 卷积后的输出图像。
    """
    # 计算输出图像的尺寸
    ho = (hi + 2 * pad - kernel) // stride + 1
    wo = (wi + 2 * pad - kernel) // stride + 1

    # 转换权重格式并对图像进行填充
    weight = np.array(weight).reshape(co, kernel, kernel, ci)
    img_pad = np.pad(img, ((pad, pad), (pad, pad), (0, 0)), "constant")
    img_out = np.zeros((ho, wo, co))


    # 使用 vdot 来优化乘加操作（MAC）
    for co_ in range(co):
        for ho_ in range(ho):
            in_h_origin = ho_ * stride - pad
            for wo_ in range(wo):
                in_w_origin = wo_ * stride - pad
                filter_h_start = max(0, -in_h_origin)
                filter_w_start = max(0, -in_w_origin)
                filter_h_end = min(kernel, hi - in_h_origin)
                filter_w_end = min(kernel, wi - in_w_origin)
                acc = float(0)
                for kh_ in range(filter_h_start, filter_h_end):
                    hi_index = in_h_origin + kh_
                    for kw_ in range(filter_w_start, filter_w_end):
                        wi_index = in_w_origin + kw_
                        # use vdot to optimize MAC operation
                        acc += np.vdot(img[hi_index][wi_index], weight[co_][kh_][kw_])
                img_out[ho_][wo_][co_] = acc

    return img_out

In [ ]:
# from PIL import Image  # 用于图像处理的PIL库
# import numpy as np  # 用于数值计算的NumPy库
# import matplotlib.pyplot as plt  # 用于显示图像的matplotlib.pyplot库
# # 打开彩色图像
# color_image = Image.open('./cat.jpg' )
# # 自定义函数转换为灰度图
# img_array = np.array(color_image) # 图像转成np.array
# Visualization(img_array)

In [ ]:
import numpy as np

def conv2d(img, weight, hi, wi, ci, co, kernel, stride, pad):
    """
    2D卷积实现
    
    参数:
        img: 输入特征图，形状为(hi, wi, ci)
        weight: 卷积核权重，形状为(co, kernel, kernel, ci)
        hi, wi, ci: 输入特征图的高度、宽度和通道数
        co: 输出通道数（卷积核个数）
        kernel: 卷积核大小（假设为正方形）
        stride: 卷积步长
        pad: 填充大小
    
    返回:
        img_out: 输出特征图，形状为(ho, wo, co)
    """
    
    # 1. 计算输出特征图尺寸
    ho = (hi + 2 * pad - kernel) // stride + 1
    wo = (wi + 2 * pad - kernel) // stride + 1
    
    # 2. 重塑卷积核权重
    weight = np.array(weight).reshape(co, kernel, kernel, ci)
    
    # 3. 初始化输出特征图
    img_out = np.zeros((ho, wo, co))
    
    # 4. 对输入特征图进行填充
    if pad > 0:
        img_padded = np.pad(img, ((pad, pad), (pad, pad), (0, 0)), mode='constant')
    else:
        img_padded = img
    hi_padded, wi_padded = hi + 2 * pad, wi + 2 * pad
    
    # 5. 主卷积循环
    for co_idx in range(co):  # 遍历输出通道
        for ho_idx in range(ho):  # 遍历输出高度
            # 计算当前卷积窗口在输入特征图中的起始位置（高度方向）
            in_h_start = ho_idx * stride
            
            for wo_idx in range(wo):  # 遍历输出宽度
                # 计算当前卷积窗口在输入特征图中的起始位置（宽度方向）
                in_w_start = wo_idx * stride
                
                # 初始化累加器
                acc = 0.0
                
                # 遍历卷积核高度方向
                for kh in range(kernel):
                    hi_index = in_h_start + kh
                    
                    # 检查索引是否越界
                    if hi_index < 0 or hi_index >= hi_padded:
                        continue
                    
                    # 遍历卷积核宽度方向
                    for kw in range(kernel):
                        wi_index = in_w_start + kw
                        
                        # 检查索引是否越界
                        if wi_index < 0 or wi_index >= wi_padded:
                            continue
                        
                        # 遍历输入通道
                        for ci_idx in range(ci):
                            # 获取输入值和权重值
                            in_data = img_padded[hi_index, wi_index, ci_idx]
                            weight_data = weight[co_idx, kh, kw, ci_idx]
                            
                            # 乘累加操作
                            acc += in_data * weight_data
                
                # 将计算结果存储到输出特征图
                img_out[ho_idx, wo_idx, co_idx] = acc
    
    return img_out


# 优化版本：使用向量化操作提高性能
def conv2d_vectorized(img, weight, hi, wi, ci, co, kernel, stride, pad):
    """
    向量化版本的2D卷积实现
    
    参数: 同conv2d函数
    返回: 同conv2d函数
    """
    
    # 1. 计算输出特征图尺寸
    ho = (hi + 2 * pad - kernel) // stride + 1
    wo = (wi + 2 * pad - kernel) // stride + 1
    
    # 2. 重塑卷积核权重
    weight = np.array(weight).reshape(co, kernel, kernel, ci)
    
    # 3. 对输入特征图进行填充
    if pad > 0:
        img_padded = np.pad(img, ((pad, pad), (pad, pad), (0, 0)), mode='constant')
    else:
        img_padded = img
    
    # 4. 使用im2col技巧将卷积操作转换为矩阵乘法
    # 获取滑动窗口的位置
    windows = np.lib.stride_tricks.sliding_window_view(
        img_padded, (kernel, kernel, ci)
    )
    
    # 选择步长为stride的窗口
    windows = windows[::stride, ::stride, :]
    
    # 调整形状以便进行矩阵乘法
    windows_flat = windows.reshape(ho * wo, kernel * kernel * ci)
    weight_flat = weight.reshape(co, kernel * kernel * ci)
    
    # 5. 执行矩阵乘法
    output_flat = np.dot(windows_flat, weight_flat.T)
    
    # 6. 调整形状为输出特征图
    img_out = output_flat.reshape(ho, wo, co)
    
    return img_out


# 测试函数
def test_conv2d():
    """测试卷积函数"""
    
    # 创建测试数据
    np.random.seed(42)
    
    # 输入特征图: 5x5x3
    hi, wi, ci = 5, 5, 3
    img = np.random.randn(hi, wi, ci)
    
    # 卷积参数
    co = 4        # 输出通道数
    kernel = 3    # 卷积核大小
    stride = 1    # 步长
    pad = 1       # 填充
    
    # 卷积核权重
    weight = np.random.randn(co, kernel, kernel, ci)
    
    print("测试卷积函数...")
    print(f"输入特征图形状: {img.shape}")
    print(f"卷积核权重形状: {weight.shape}")
    print(f"卷积参数: kernel={kernel}, stride={stride}, pad={pad}")
    
    # 计算输出尺寸
    ho = (hi + 2 * pad - kernel) // stride + 1
    wo = (wi + 2 * pad - kernel) // stride + 1
    print(f"输出特征图形状: ({ho}, {wo}, {co})")
    
    # 测试基本卷积函数
    print("\n1. 测试基本卷积函数:")
    output_basic = conv2d(img, weight.flatten(), hi, wi, ci, co, kernel, stride, pad)
    print(f"输出形状: {output_basic.shape}")
    print(f"输出示例值（第一个通道的前2x2区域）:")
    print(output_basic[:2, :2, 0])
    
    # 测试向量化卷积函数
    print("\n2. 测试向量化卷积函数:")
    output_vectorized = conv2d_vectorized(img, weight.flatten(), hi, wi, ci, co, kernel, stride, pad)
    print(f"输出形状: {output_vectorized.shape}")
    print(f"输出示例值（第一个通道的前2x2区域）:")
    print(output_vectorized[:2, :2, 0])
    
    # 验证两个函数的结果是否一致
    print("\n3. 验证两个函数结果的一致性:")
    max_diff = np.max(np.abs(output_basic - output_vectorized))
    print(f"最大差异: {max_diff}")
    if max_diff < 1e-10:
        print("✓ 两个函数的结果一致")
    else:
        print("✗ 两个函数的结果不一致")
    
    # 性能对比
    print("\n4. 性能对比:")
    import time
    
    # 基本卷积函数的性能
    start_time = time.time()
    for _ in range(10):
        conv2d(img, weight.flatten(), hi, wi, ci, co, kernel, stride, pad)
    basic_time = time.time() - start_time
    
    # 向量化卷积函数的性能
    start_time = time.time()
    for _ in range(10):
        conv2d_vectorized(img, weight.flatten(), hi, wi, ci, co, kernel, stride, pad)
    vectorized_time = time.time() - start_time
    
    print(f"基本卷积函数时间: {basic_time:.4f}秒")
    print(f"向量化卷积函数时间: {vectorized_time:.4f}秒")
    print(f"速度提升: {basic_time/vectorized_time:.2f}倍")
    
    return output_basic, output_vectorized


# 可视化函数
def visualize_conv_operation():
    """可视化卷积操作"""
    
    # 创建简单的示例数据
    hi, wi, ci = 4, 4, 1
    img = np.array([
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 16]
    ]).reshape(hi, wi, ci)
    
    # 简单的卷积核
    kernel = 3
    co = 1
    weight = np.ones((co, kernel, kernel, ci)).flatten()
    
    # 执行卷积
    output = conv2d(img, weight, hi, wi, ci, co, kernel=3, stride=1, pad=0)
    
    print("卷积操作可视化示例:")
    print("\n输入特征图:")
    print(img[:, :, 0])
    
    print("\n卷积核 (3x3, 全1):")
    print(np.ones((3, 3)))
    
    print("\n输出特征图:")
    print(output[:, :, 0])
    
    # 解释卷积计算
    print("\n卷积计算解释:")
    print("对于输出位置(0,0):")
    print("  输入窗口:")
    print("    [1, 2, 3]")
    print("    [5, 6, 7]")
    print("    [9, 10, 11]")
    print(f"  计算结果: 1+2+3+5+6+7+9+10+11 = {1+2+3+5+6+7+9+10+11}")
    print(f"  实际输出: {output[0, 0, 0]}")


In [ ]:

# 运行测试
print("=" * 60)
print("卷积函数测试与演示")
print("=" * 60)

# 可视化卷积操作
visualize_conv_operation()

print("\n" + "=" * 60)
print("完整卷积函数测试")
print("=" * 60)

# 运行完整测试
test_conv2d()

# 示例用法
print("\n" + "=" * 60)
print("卷积函数使用示例")
print("=" * 60)

# 创建示例数据
hi, wi, ci = 7, 7, 3
co = 16
kernel = 3
stride = 2
pad = 1

# 随机生成输入和权重
img = np.random.randn(hi, wi, ci)
weight = np.random.randn(co, kernel, kernel, ci).flatten()

# 计算卷积
output = conv2d(img, weight, hi, wi, ci, co, kernel, stride, pad)

print(f"输入特征图形状: {img.shape}")
print(f"卷积参数: kernel={kernel}, stride={stride}, pad={pad}")
print(f"输出特征图形状: {output.shape}")

# 验证输出尺寸计算公式
ho_calc = (hi + 2 * pad - kernel) // stride + 1
wo_calc = (wi + 2 * pad - kernel) // stride + 1
print(f"\n输出尺寸验证:")
print(f"  计算得到的尺寸: ({ho_calc}, {wo_calc}, {co})")
print(f"  实际输出尺寸: {output.shape}")
print(f"  验证{'通过' if (ho_calc, wo_calc, co) == output.shape else '失败'}")